In [ ]:
import json
import shutil
from pathlib import Path

base_dir = Path("/root/autodl-tmp/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1")
all_jsonl = base_dir / "all.jsonl"
copy_jsonl = base_dir / "seal0-0623.jsonl"
backup_jsonl = base_dir / "all.jsonl.bak"

# ---------- 1. 备份原始 all.jsonl ----------
shutil.copy2(all_jsonl, backup_jsonl)
print(f"[✓] 已备份原始文件到: {backup_jsonl}")

# ---------- 2. 读取 bamboogle_data_copy.jsonl，建立 id → Question 映射 ----------
id_to_question = {}
with open(copy_jsonl, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        # 兼容不同大小写的字段名
        qid = item.get("id") or item.get("ID") or item.get("Id")
        question = item.get("Question") or item.get("question") or item.get("query")
        if qid is None:
            print(f"[!] 第 {line_no} 行没有 id 字段，跳过")
            continue
        if question is None:
            print(f"[!] 第 {line_no} 行 (id={qid}) 没有 Question 字段，跳过")
            continue
        id_to_question[qid] = question

print(f"[✓] 从 {copy_jsonl.name} 读取到 {len(id_to_question)} 条 id→Question 映射")

# ---------- 3. 读取 all.jsonl，补充 Question，写回 ----------
updated_count = 0
not_found_count = 0
already_has_count = 0
records = []

with open(all_jsonl, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        qid = item.get("id") or item.get("ID") or item.get("Id")

        # 如果已经有 Question 字段就跳过（不覆盖）
        if "Question" in item or "question" in item:
            already_has_count += 1
            records.append(item)
            continue

        if qid is not None and qid in id_to_question:
            # 在原有字段后面追加 Question，不改动已有字段
            item["Question"] = id_to_question[qid]
            updated_count += 1
        else:
            not_found_count += 1
            if line_no <= 5:  # 只打印前几条警告
                print(f"[!] all.jsonl 第 {line_no} 行 (id={qid}) 在映射表中未找到对应 Question")

        records.append(item)

# ---------- 4. 写回 all.jsonl ----------
with open(all_jsonl, "w", encoding="utf-8") as f:
    for item in records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n========== 处理完成 ==========")
print(f"  总记录数:           {len(records)}")
print(f"  本次新增 Question:  {updated_count}")
print(f"  原本已有 Question:  {already_has_count}")
print(f"  未找到匹配 id:      {not_found_count}")
print(f"  备份文件:           {backup_jsonl}")
print(f"================================")


[✓] 已备份原始文件到: /root/autodl-tmp/project/Search-o1/outputs/runs.baselines/bamboogle.qwen3.5-9b.search_o1/all.jsonl.bak


JSONDecodeError: Expecting value: line 1 column 2 (char 1)

In [1]:
import json
import shutil
from pathlib import Path

base_dir = Path("/root/autodl-tmp/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1")
all_jsonl = base_dir / "all.jsonl"
copy_jsonl = base_dir / "seal0-0623.jsonl"
backup_jsonl = base_dir / "all.jsonl.bak"

# ========== 1. 读取 bamboogle_data_copy.jsonl（JSON 数组格式） ==========
print(f"【读取 {copy_jsonl}】")
with open(copy_jsonl, "r", encoding="utf-8") as f:
    first_char = f.read(1)  # 先看第一个字符

if first_char == '[':
    # 整个文件是一个 JSON 数组
    print(f"  → 检测到文件是 JSON 数组格式（以 '[' 开头），用 json.load() 整体读取")
    with open(copy_jsonl, "r", encoding="utf-8") as f:
        copy_records = json.load(f)
    print(f"  ✅ 成功读取 {len(copy_records)} 条记录")
else:
    # 标准 JSONL，逐行读取
    print(f"  → 文件是标准 JSONL 格式，逐行读取")
    copy_records = []
    with open(copy_jsonl, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            copy_records.append(item)
    print(f"  ✅ 成功读取 {len(copy_records)} 条记录")

# 建立 id → Question 映射
id_to_question = {}
for item in copy_records:
    qid = item.get("id") or item.get("ID") or item.get("Id")
    question = item.get("Question") or item.get("question") or item.get("query")
    if qid is not None and question is not None:
        id_to_question[qid] = question

print(f"  ✅ 建立 {len(id_to_question)} 条 id→Question 映射")

# ========== 2. 读取 all.jsonl（标准 JSONL 格式） ==========
print("\n【读取 all.jsonl】")
all_records = []
with open(all_jsonl, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        all_records.append(item)

print(f"  ✅ 成功读取 {len(all_records)} 条记录")

# ========== 3. 补充 Question 字段 ==========
print("\n【补充 Question 字段】")
updated_count = 0
not_found_count = 0
already_has_count = 0

for i, item in enumerate(all_records):
    qid = item.get("id") or item.get("ID") or item.get("Id")

    # 如果已经有 Question 字段就跳过（不覆盖）
    if "Question" in item or "question" in item:
        already_has_count += 1
        continue

    if qid is not None and qid in id_to_question:
        item["Question"] = id_to_question[qid]
        updated_count += 1
    else:
        not_found_count += 1
        print(f"  [!] 第 {i+1} 行 (id={qid}) 未找到匹配的 Question")

# ========== 4. 备份并写回 ==========
if not backup_jsonl.exists():
    shutil.copy2(all_jsonl, backup_jsonl)
    print(f"\n✅ 已备份原始文件到: {backup_jsonl}")
else:
    print(f"\n✅ 备份文件已存在: {backup_jsonl}（跳过备份）")

with open(all_jsonl, "w", encoding="utf-8") as f:
    for item in all_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n{'=' * 60}")
print(f"  处理完成!")
print(f"  总记录数:           {len(all_records)}")
print(f"  本次新增 Question:  {updated_count}")
print(f"  原本已有 Question:  {already_has_count}")
print(f"  未找到匹配 id:      {not_found_count}")
print(f"{'=' * 60}")


【读取 /root/autodl-tmp/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/seal0-0623.jsonl】
  → 文件是标准 JSONL 格式，逐行读取
  ✅ 成功读取 111 条记录
  ✅ 建立 111 条 id→Question 映射

【读取 all.jsonl】
  ✅ 成功读取 111 条记录

【补充 Question 字段】

✅ 已备份原始文件到: /root/autodl-tmp/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/all.jsonl.bak

  处理完成!
  总记录数:           111
  本次新增 Question:  111
  原本已有 Question:  0
  未找到匹配 id:      0
